# Level 2 — Preprocessing teks Indonesia untuk klasifikasi Emergency / Non-Emergency

> **Data sintetis.** Seluruh teks di notebook ini dibangkitkan dari templat atau ditulis tangan
> (lihat `../disaster-classification/DATA_CARD.md`). Angka di sini hanya menunjukkan bahwa kodenya
> bekerja, bukan performa pada data nyata.

**Cara menjalankan:** jalankan sel dari atas ke bawah. Prasyarat: `pip install -r requirements-ml.txt`
dari folder utama repository. Notebook ini membaca data lewat jalur relatif dan tidak mengunduh apa pun.

In [1]:
import sys
import warnings
from pathlib import Path

# Notebook ini berada di examples/<folder-level>/. Paket nlp_hub ada di examples/.
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", message="'count' is passed as positional argument")

## 1. Memuat dataset sintetis
`muat_dataset()` membaca CSV dan membuang duplikat persis.

In [2]:
from nlp_hub.data import muat_dataset

df = muat_dataset()
print(df.shape)
df[["event_id", "post_type", "label", "text"]].head(5)

(564, 6)


,event_id,post_type,label,text
0,E079,edukasi,Non-Emergency,Tips menghadapi banjir: kenali jalur evakuasi ...
1,E070,peringatan,Non-Emergency,"Musim pancaroba sudah dekat, waspada gempa ya ..."
2,E061,laporan,Emergency,"Setelah gempa tadi, atap ruko di Gang Flamboya..."
3,E040,tidak_terkait,Non-Emergency,"Konser semalam gila, penonton berteriak sampai..."
4,E032,laporan,Emergency,Jalan menuju Kelurahan Bumi Indah tertutup lon...


## 2. Menelusuri satu postingan langkah demi langkah
Setiap langkah mengambil hasil langkah sebelumnya. Urutan penting: hashtag diproses sebelum huruf
menjadi kecil, dan emoji sebelum tokenisasi.

In [3]:
from nlp_hub.preprocessing import (
    ganti_mention, ganti_url, kurangi_huruf_berulang, normalisasi_unicode, proses_emoji,
    proses_hashtag, case_folding, tokenisasi,
)

teks = "@bpbd_contoh TOLOOONG banjir naik! #SiagaBencana https://contoh.example/1 🙏"
langkah = normalisasi_unicode(teks); print("1 unicode  :", langkah)
langkah = ganti_mention(ganti_url(langkah)); print("2 url/@    :", " ".join(langkah.split()))
langkah = proses_hashtag(langkah, "kata"); print("3 hashtag  :", " ".join(langkah.split()))
langkah = proses_emoji(langkah, "kata"); print("4 emoji    :", " ".join(langkah.split()))
langkah = case_folding(langkah); print("5 huruf kcl:", " ".join(langkah.split()))
langkah = kurangi_huruf_berulang(langkah); print("6 berulang :", " ".join(langkah.split()))
print("7 token    :", tokenisasi(langkah))

1 unicode  : @bpbd_contoh TOLOOONG banjir naik! #SiagaBencana https://contoh.example/1 🙏
2 url/@    : tok_mention TOLOOONG banjir naik! #SiagaBencana tok_url 🙏
3 hashtag  : tok_mention TOLOOONG banjir naik! Siaga Bencana tok_url 🙏
4 emoji    : tok_mention TOLOOONG banjir naik! Siaga Bencana tok_url emo_doa
5 huruf kcl: tok_mention tolooong banjir naik! siaga bencana tok_url emo_doa
6 berulang : tok_mention tolong banjir naik! siaga bencana tok_url emo_doa
7 token    : ['tok_mention', 'tolong', 'banjir', 'naik', 'siaga', 'bencana', 'tok_url', 'emo_doa']


## 3. Fungsi gabungan dan konfigurasi
`praproses_token` menjalankan langkah yang sama; pilihan diatur lewat `KonfigurasiPraproses`.
Hasilnya harus sama dengan langkah manual di atas.

In [4]:
from nlp_hub.preprocessing import KonfigurasiPraproses, praproses, praproses_token

print(praproses_token(teks))
lengkap = KonfigurasiPraproses(angka_waktu_lokasi=True, negasi=True, stopword=True, stem=True)
print(praproses("Tidak ada listrik di Jl. Melati RT 08 pukul 08.12, tolong!", lengkap))

['tok_mention', 'tolong', 'banjir', 'naik', 'siaga', 'bencana', 'tok_url', 'emo_doa']
tidak neg_ada listrik tok_lokasi tok_waktu tolong


## 4. Efek pada ukuran kosakata
Normalisasi, stemming, dan penggantian lokasi menyatukan beberapa bentuk. Kita ukur jumlah *tipe* (kata unik).
Pengurangan kosakata **bukan** bukti bahwa model menjadi lebih baik; itu harus diuji (lihat halaman *Perbandingan Pipeline*).

In [5]:
import pandas as pd

konfigurasi = {
    "tanpa normalisasi": KonfigurasiPraproses(informal=False, huruf_berulang=False),
    "dasar": KonfigurasiPraproses(),
    "+ angka/waktu/lokasi": KonfigurasiPraproses(angka_waktu_lokasi=True),
    "+ stopword konservatif": KonfigurasiPraproses(stopword="konservatif"),
    "+ stemming": KonfigurasiPraproses(stem=True),
}
baris = []
for nama, k in konfigurasi.items():
    token = [t for x in df["text"] for t in praproses_token(x, k)]
    baris.append({"konfigurasi": nama, "token": len(token), "tipe": len(set(token))})
pd.DataFrame(baris)

,konfigurasi,token,tipe
0,tanpa normalisasi,7124,685
1,dasar,7124,596
2,+ angka/waktu/lokasi,6393,554
3,+ stopword konservatif,6405,580
4,+ stemming,7124,537


## 5. Audit: apakah ada teks yang menjadi kosong?
Stopword bawaan Sastrawi membuang kata seperti `tidak`, `belum`, dan `tolong`. Periksa kata yang terbuang dari satu contoh.

In [6]:
from nlp_hub.preprocessing import stopwords_sastrawi

contoh = "Tolong, air belum surut dan listrik tidak ada"
terbuang = [t for t in praproses_token(contoh) if t in stopwords_sastrawi()]
print("kata yang dibuang Sastrawi:", terbuang)
kosong = sum(1 for x in df["text"] if not praproses(x, KonfigurasiPraproses(stopword="sastrawi")).strip())
print("teks yang menjadi kosong:", kosong)

kata yang dibuang Sastrawi: ['tolong', 'belum', 'dan', 'tidak', 'ada']
teks yang menjadi kosong: 0


## Latihan
1. Tambahkan konfigurasi baru ke sel 4 (mis. `negasi=True`) dan bandingkan jumlah tipe.
2. Ubah `contoh` pada sel 5 menjadi pasangan kalimat berlawanan (`Ada korban` / `Tidak ada korban`) dan bandingkan hasil dengan stopword Sastrawi dan konservatif.
3. Baca halaman *Perbandingan Pipeline* lalu jalankan `10_perbandingan_pipeline.py`. Apakah kesimpulannya berubah jika Anda mengganti seed?

**Keterbatasan:** dataset sintetis dan kecil; tidak ada kesimpulan tentang data Twitter/X nyata.